# Project 1

## Reading in the data

I'm using pandas to load the data: "NYC Leading Causes of Death.csv" dataset. This dataset is found on NYC Open Data (https://data.cityofnewyork.us/Health/New-York-City-Leading-Causes-of-Death/jb7j-dtam/about_data)

In [5]:
import pandas as pd
# Read the CSV file
df = pd.read_csv("NYC Leading Causes of Death.csv")

# Preview the data
df.head()

,Year,Leading Cause,Sex,Race Ethnicity,Deaths,Death Rate,Age Adjusted Death Rate
0,2021,"Diseases of Heart (I00-I09, I11, I13, I20-I51)",Male,Not Stated/Unknown,190,NaN,NaN
1,2021,Alzheimer's Disease (G30),Female,Not Stated/Unknown,7,NaN,NaN
2,2021,"Diseases of Heart (I00-I09, I11, I13, I20-I51)",Female,Not Stated/Unknown,113,NaN,NaN
3,2021,Malignant Neoplasms (Cancer: C00-C97),Male,Not Stated/Unknown,84,NaN,NaN
4,2021,Cerebrovascular Disease (Stroke: I60-I69),Male,Other Race/ Ethnicity,11,NaN,NaN


## Filtering data
I will focus on the "Deaths" numeric column

In [9]:
# Print out the Deaths column
print(df["Deaths"])

# Convert Deaths from object to float
df["Deaths"] = pd.to_numeric(df["Deaths"], errors="coerce")

# Check the result
print(df["Deaths"].dtype)
print(df["Deaths"].head())

0       190.0
1         7.0
2       113.0
3        84.0
4        11.0
        ...  
2097    528.0
2098     24.0
2099    213.0
2100      5.0
2101     27.0
Name: Deaths, Length: 2102, dtype: float64
float64
0    190.0
1      7.0
2    113.0
3     84.0
4     11.0
Name: Deaths, dtype: float64


## Compute Mean, Median, and Mode
Computing the mean, median, and mode for the number of Deaths for each leading causes at NYC


In [10]:
# Mean
mean_deaths = df["Deaths"].mean()
print("Mean deaths:", mean_deaths)

# Median
median_deaths = df["Deaths"].median()
print("Median deaths:", median_deaths)

# Mode
mode_deaths = df["Deaths"].mode()
print("Mode deaths:")
print(mode_deaths)


Mean deaths: 429.2561099796334
Median deaths: 140.0
Mode deaths:
0    1.0
Name: Deaths, dtype: float64


## Hardway

In [13]:
import csv

deaths = []

with open("NYC Leading Causes of Death.csv", newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    header = next(reader) 
    print("Header row:", header)
    
    # finding the index part of the 'Deaths' column
    deaths_idx = header.index("Deaths")
    
    for row in reader:
        value = row[deaths_idx]
        if value == "" or value == ".":
            continue  # sorting data by skipping empty values
        
        # convert values to float
        value = value.replace(",", "")
        num = float(value)
        deaths.append(num)

print("Number of death values:", len(deaths))
print("First 10 values:", deaths[:10])


Header row: ['Year', 'Leading Cause', 'Sex', 'Race Ethnicity', 'Deaths', 'Death Rate', 'Age Adjusted Death Rate']
Number of death values: 1964
First 10 values: [190.0, 7.0, 113.0, 84.0, 11.0, 14.0, 66.0, 15.0, 52.0, 78.0]


## Computing Mean, Median, Mode with the hard way

In [20]:
total = 0
count = 0

for d in deaths:
    total += d
    count += 1

# Median of deaths
mean_deaths_hard = total / count
print("Mean deaths (hard way):", mean_deaths_hard)

#Median of deaths
sorted_deaths = sorted(deaths) # sort the list
n = len(sorted_deaths)

if n % 2 == 1:
    # checking if the list is odd number
    median_deaths_hard = sorted_deaths[n // 2]
else:
    # finding median if the list is even number
    mid1 = sorted_deaths[n // 2 - 1]
    mid2 = sorted_deaths[n // 2]
    median_deaths_hard = (mid1 + mid2) / 2

print("Median deaths (hard way):", median_deaths_hard)

# Mode of deaths
counts = {}

# count frequencies
for d in deaths:
    if d in counts:
        counts[d] += 1
    else:
        counts[d] = 1

# finding the highest frequency
max_count = 0
for freq in counts.values():
    if freq > max_count:
        max_count = freq

# collecing all values with highest frequency
modes_deaths_hard = []
for value, freq in counts.items():
    if freq == max_count:
        modes_deaths_hard.append(value)

print("Mode deaths (hard way):", modes_deaths_hard)
print("Max frequency:", max_count)


Mean deaths (hard way): 429.2561099796334
Median deaths (hard way): 140.0
Mode deaths (hard way): [1.0]
Max frequency: 61


## Data Visualization

In [25]:
# Aggregating Deaths by Year

import pandas as pd

# Load data
df = pd.read_csv("NYC Leading Causes of Death.csv")

# Making Deaths numeric
df["Deaths"] = pd.to_numeric(
    df["Deaths"].astype(str).str.replace(",", ""),
    errors="coerce"
)
df = df.dropna(subset=["Deaths"])

# Total deaths per year
deaths_by_year = (
    df.groupby("Year")["Deaths"]
      .sum()
      .sort_index()
)

deaths_by_year


Year
2007    53996.0
2008    54138.0
2009    52820.0
2010    52505.0
2011    52726.0
2012    52420.0
2013    53387.0
2014    53006.0
2015    54120.0
2016    54280.0
2017    54319.0
2018    55081.0
2019    54559.0
2020    82142.0
2021    63560.0
Name: Deaths, dtype: float64

In [27]:
# Convert to Python data
years = list(deaths_by_year.index)
totals = [int(v) for v in deaths_by_year.values]

print(years)
print(totals[:5]) #Checking data output


[2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]
[53996, 54138, 52820, 52505, 52726]


In [29]:
# Text Sparkline rounded to the nearest thousand

THOUSANDS_PER_STAR = 1_000   # 1,000 deaths per star

print("NYC deaths by year (in thousands)\n")
print(f"(Each '*' represents about {THOUSANDS_PER_STAR:,} deaths.)\n")

for year, total in zip(years, totals):
    # integer thousands and remainder
    base_thousands = total // THOUSANDS_PER_STAR
    remainder = total % THOUSANDS_PER_STAR

    # if we are close to the next thousand (>= 500), round up by 1
    if remainder >= THOUSANDS_PER_STAR / 2:
        base_thousands += 1

    star_count = int(base_thousands)

    # show at least one star for any nonzero value
    if star_count == 0 and total > 0:
        star_count = 1

    stars = "*" * star_count
    print(f"{year}: {stars}")


NYC deaths by year (in thousands)

(Each '*' represents about 1,000 deaths.)

2007: ******************************************************
2008: ******************************************************
2009: *****************************************************
2010: *****************************************************
2011: *****************************************************
2012: ****************************************************
2013: *****************************************************
2014: *****************************************************
2015: ******************************************************
2016: ******************************************************
2017: ******************************************************
2018: *******************************************************
2019: *******************************************************
2020: **********************************************************************************
2021: ***************************************